In [ ]:
# Install required libraries (Colab already has most)
!pip install tensorflow scikit-learn matplotlib pandas numpy -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import warnings
warnings.filterwarnings('ignore')

tf.random.set_seed(42)
np.random.seed(42)
print('TensorFlow version:', tf.__version__)
print('All libraries loaded successfully ✓')

In [ ]:
def generate_ola_dataset(start='2023-01-01', periods=8760):
    """
    Generate synthetic hourly Ola bike ride demand data.
    Returns a DataFrame with timestamp and ride_count columns.
    """
    timestamps = pd.date_range(start=start, periods=periods, freq='H')
    ride_counts = []

    for ts in timestamps:
        hour = ts.hour
        day = ts.dayofweek   # 0=Mon, 6=Sun
        month = ts.month
        is_weekend = day >= 5

        # Seasonal multiplier
        season_map = {12: 0.85, 1: 0.85, 2: 0.90,   # Winter
                      3: 1.00, 4: 1.05, 5: 1.10,    # Spring
                      6: 1.15, 7: 1.10, 8: 1.05,    # Summer
                      9: 0.80, 10: 0.75, 11: 0.80}  # Monsoon
        season_mult = season_map[month]

        # Hourly base demand
        if 0 <= hour <= 5:
            base = 6
        elif 6 <= hour <= 9:
            base = 55 if not is_weekend else 25
        elif 10 <= hour <= 16:
            base = 30 if not is_weekend else 35
        elif 17 <= hour <= 20:
            base = 62 if not is_weekend else 42
        else:
            base = 18

        # Add noise and apply seasonal multiplier
        noise = np.random.normal(0, 4)
        count = max(0, int(base * season_mult + noise))
        ride_counts.append(count)

    df = pd.DataFrame({'timestamp': timestamps, 'ride_count': ride_counts})
    df['hour'] = df['timestamp'].dt.hour
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    df['month'] = df['timestamp'].dt.month
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    df.set_index('timestamp', inplace=True)
    return df

df = generate_ola_dataset()
print('Dataset shape:', df.shape)
print(df.head(10))
print('\nBasic Statistics:')
print(df['ride_count'].describe())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Ola Bike Ride Demand — Exploratory Analysis', fontsize=14, fontweight='bold')

# 1. One week of ride demand
week_data = df['ride_count'][:168]
axes[0, 0].plot(week_data.values, color='#185FA5', linewidth=1.2)
axes[0, 0].set_title('One Week of Ride Demand (Hourly)')
axes[0, 0].set_xlabel('Hour')
axes[0, 0].set_ylabel('Ride Count')
axes[0, 0].set_xticks(range(0, 168, 24))
axes[0, 0].set_xticklabels(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'])
axes[0, 0].grid(True, alpha=0.3)

# 2. Average demand by hour
hourly_avg = df.groupby('hour')['ride_count'].mean()
axes[0, 1].bar(hourly_avg.index, hourly_avg.values, color='#185FA5', alpha=0.8)
axes[0, 1].set_title('Average Ride Demand by Hour of Day')
axes[0, 1].set_xlabel('Hour')
axes[0, 1].set_ylabel('Avg Ride Count')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# 3. Average demand by day of week
daily_avg = df.groupby('day_of_week')['ride_count'].mean()
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
axes[1, 0].bar(days, daily_avg.values, color='#D85A30', alpha=0.8)
axes[1, 0].set_title('Average Ride Demand by Day of Week')
axes[1, 0].set_xlabel('Day')
axes[1, 0].set_ylabel('Avg Ride Count')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# 4. Monthly demand
monthly_avg = df.groupby('month')['ride_count'].mean()
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[1, 1].bar(months, monthly_avg.values, color='#1D9E75', alpha=0.8)
axes[1, 1].set_title('Average Ride Demand by Month (Seasonal)')
axes[1, 1].set_xlabel('Month')
axes[1, 1].set_ylabel('Avg Ride Count')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('ola_eda.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA plots saved ✓')

In [ ]:
SEQ_LEN = 24   # lookback window (hours)

# Scale ride counts
scaler = MinMaxScaler(feature_range=(0, 1))
ride_scaled = scaler.fit_transform(df[['ride_count']]).flatten()

def create_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i + seq_len])
        y.append(data[i + seq_len])
    return np.array(X), np.array(y)

X, y = create_sequences(ride_scaled, SEQ_LEN)

# Train / Val / Test split
n = len(X)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

X_train, y_train = X[:train_end], y[:train_end]
X_val,   y_val   = X[train_end:val_end], y[train_end:val_end]
X_test,  y_test  = X[val_end:], y[val_end:]

# Reshape for LSTM: (samples, timesteps, features)
X_train = X_train.reshape(-1, SEQ_LEN, 1)
X_val   = X_val.reshape(-1,   SEQ_LEN, 1)
X_test  = X_test.reshape(-1,  SEQ_LEN, 1)

print(f'Training samples   : {X_train.shape[0]}')
print(f'Validation samples : {X_val.shape[0]}')
print(f'Test samples       : {X_test.shape[0]}')
print(f'Input shape        : {X_train.shape}')

Build the Stacked LSTM Model

Architecture:
```
Input (24, 1)
  → LSTM(128, return_sequences=True)
  → Dropout(0.2)
  → LSTM(64, return_sequences=False)
  → Dropout(0.2)
  → Dense(32, activation='relu')
  → Dense(1, activation='linear')  ← predicted ride count
```

In [ ]:
def build_lstm_model(seq_len):
    model = Sequential([
        LSTM(128, input_shape=(seq_len, 1), return_sequences=True),
        Dropout(0.2),
        LSTM(64, return_sequences=False),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1, activation='linear')
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    return model

model = build_lstm_model(SEQ_LEN)
model.summary()

## 6. Train the Model

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1)
reduce_lr  = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print('\nTraining complete ✓')
print(f'Best val_loss: {min(history.history["val_loss"]):.5f}')

## 7. Training Loss Curves

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'],     label='Train Loss', color='#185FA5', linewidth=1.5)
plt.plot(history.history['val_loss'], label='Val Loss',   color='#D85A30', linewidth=1.5, linestyle='--')
plt.title('Training & Validation Loss (MSE) over Epochs', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_loss.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Evaluate on Test Set

In [ ]:
# Predict
y_pred_scaled = model.predict(X_test).flatten()

# Inverse transform
y_pred = scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
y_true = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

# Metrics
mse  = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_true, y_pred)
mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100

print('='*40)
print('     TEST SET EVALUATION METRICS')
print('='*40)
print(f'  MSE  : {mse:.4f}')
print(f'  RMSE : {rmse:.4f}')
print(f'  MAE  : {mae:.4f}')
print(f'  MAPE : {mape:.2f}%')
print('='*40)

## 9. Actual vs Predicted — Visualization

In [ ]:
plot_n = 168  # show last 7 days

plt.figure(figsize=(14, 5))
plt.plot(y_true[:plot_n],  label='Actual',         color='#185FA5', linewidth=1.5)
plt.plot(y_pred[:plot_n],  label='LSTM Predicted', color='#D85A30', linewidth=1.5, linestyle='--')
plt.title('Actual vs LSTM Predicted Ride Demand (Test Set — 7 Days)', fontweight='bold')
plt.xlabel('Hour')
plt.ylabel('Ride Count')
plt.xticks(range(0, plot_n, 24), ['Day '+str(i+1) for i in range(7)])
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Interactive Ride Demand Predictor (ipywidgets)

Use the sliders and dropdowns below to predict ride demand for any time condition.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

hour_slider   = widgets.IntSlider(value=8, min=0, max=23, description='Hour:', style={'description_width': '80px'})
day_dropdown  = widgets.Dropdown(options=[('Monday',0),('Tuesday',1),('Wednesday',2),('Thursday',3),('Friday',4),('Saturday',5),('Sunday',6)], description='Day:', style={'description_width': '80px'})
month_slider  = widgets.IntSlider(value=6, min=1, max=12, description='Month:', style={'description_width': '80px'})
predict_btn   = widgets.Button(description='Predict Demand', button_style='primary', icon='motorcycle')
output_area   = widgets.Output()

def on_predict(b):
    with output_area:
        clear_output(wait=True)
        hour  = hour_slider.value
        day   = day_dropdown.value
        month = month_slider.value
        is_weekend = day >= 5

        # Build a 24-hour input sequence mimicking the selected conditions
        season_map = {12:0.85,1:0.85,2:0.90,3:1.00,4:1.05,5:1.10,
                      6:1.15,7:1.10,8:1.05,9:0.80,10:0.75,11:0.80}
        sm = season_map[month]

        def base_for_hour(h):
            if 0 <= h <= 5:   return 6
            elif 6 <= h <= 9: return 55 if not is_weekend else 25
            elif 10 <= h <= 16: return 30 if not is_weekend else 35
            elif 17 <= h <= 20: return 62 if not is_weekend else 42
            else: return 18

        seq_raw = np.array([max(0, base_for_hour((hour - 24 + i) % 24) * sm + np.random.normal(0,2))
                            for i in range(24)])
        seq_scaled = scaler.transform(seq_raw.reshape(-1, 1)).flatten()
        X_input = seq_scaled.reshape(1, SEQ_LEN, 1)

        pred_scaled = model.predict(X_input, verbose=0)[0][0]
        pred_count  = int(scaler.inverse_transform([[pred_scaled]])[0][0])

        level = 'Low' if pred_count < 15 else 'Moderate' if pred_count < 40 else 'High (Surge likely)'
        day_names = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
        month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

        print('='*45)
        print('       LSTM RIDE DEMAND PREDICTION')
        print('='*45)
        print(f'  Time   : {hour:02d}:00  |  {day_names[day]}')
        print(f'  Month  : {month_names[month-1]}')
        print(f'  Rides  : {pred_count} requests/hour')
        print(f'  Level  : {level}')
        print('='*45)

        # Mini bar chart
        fig, ax = plt.subplots(figsize=(8, 3))
        next_hours = [(hour + i) % 24 for i in range(6)]
        next_preds = []
        for nh in next_hours:
            s = np.array([max(0, base_for_hour((nh - 24 + j) % 24) * sm + np.random.normal(0,2))
                          for j in range(24)])
            s_sc = scaler.transform(s.reshape(-1,1)).flatten().reshape(1, SEQ_LEN, 1)
            p = model.predict(s_sc, verbose=0)[0][0]
            next_preds.append(int(scaler.inverse_transform([[p]])[0][0]))
        ax.bar([f'{h:02d}:00' for h in next_hours], next_preds, color='#185FA5', alpha=0.85)
        ax.set_title('6-Hour Ahead Forecast', fontweight='bold')
        ax.set_ylabel('Predicted Rides')
        ax.grid(True, alpha=0.3, axis='y')
        plt.tight_layout()
        plt.show()

predict_btn.on_click(on_predict)
display(widgets.VBox([hour_slider, day_dropdown, month_slider, predict_btn, output_area]))

## 11. Save the Model

Save the trained LSTM model for later use or deployment.

In [ ]:
model.save('ola_lstm_model.h5')
print('Model saved as ola_lstm_model.h5 ✓')

# To load later:
# from tensorflow.keras.models import load_model
# model = load_model('ola_lstm_model.h5')